In [1]:
import os
import sys
import pickle
import numpy as np
import tiktoken

from sklearn.metrics.pairwise import cosine_similarity

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from langchain_huggingface import (
    HuggingFaceEmbeddings
)

from langchain_groq import ChatGroq


class PDFChatbot:

    def __init__(self, pdf_path, groq_api_key):

        self.pdf_path = "/Users/Aryaanil/Documents/survey cdc.pdf"

        self.groq_api_key = "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"

        self.history = []

        self.last_sources = []

        self.token_limit = 3000

        self.index_file = (
            os.path.splitext(pdf_path)[0]
            + "_index.pkl"
        )

        self.tokenizer = tiktoken.get_encoding(
            "cl100k_base"
        )

        self.embedding_model = (
            HuggingFaceEmbeddings(
                model_name=(
                    "sentence-transformers/"
                    "all-MiniLM-L6-v2"
                )
            )
        )

        self.load_llm()

        self.load_or_create_index()


    def load_llm(self):

        self.llm = ChatGroq(
            groq_api_key=self.groq_api_key,
            model_name="llama-3.3-70b-versatile"
        )


    def load_or_create_index(self):

        if os.path.exists(self.index_file):

            print(
                f"\nLoading saved index:"
                f" {self.index_file}"
            )

            with open(
                self.index_file,
                "rb"
            ) as file:

                data = pickle.load(file)

                self.texts = data["texts"]

                self.metadata = data["metadata"]

                self.document_embeddings = (
                    data["embeddings"]
                )

        else:

            print(
                "\nNo saved index found."
            )

            print(
                "Creating new index..."
            )

            loader = PyPDFLoader(
                self.pdf_path
            )

            documents = loader.load()

            splitter = (
                RecursiveCharacterTextSplitter(
                    chunk_size=500,
                    chunk_overlap=50
                )
            )

            chunks = splitter.split_documents(
                documents
            )

            self.texts = [
                doc.page_content
                for doc in chunks
            ]

            self.metadata = [
                doc.metadata
                for doc in chunks
            ]

            embeddings = (
                self.embedding_model
                .embed_documents(
                    self.texts
                )
            )

            self.document_embeddings = (
                np.array(embeddings)
            )

            with open(
                self.index_file,
                "wb"
            ) as file:

                pickle.dump(
                    {
                        "texts": self.texts,
                        "metadata": self.metadata,
                        "embeddings":
                        self.document_embeddings
                    },
                    file
                )

            print(
                f"Index saved as:"
                f" {self.index_file}"
            )


    def retrieve(self, query, k=3):

        query_embedding = (
            self.embedding_model
            .embed_query(query)
        )

        similarity_scores = (
            cosine_similarity(
                [query_embedding],
                self.document_embeddings
            )[0]
        )

        top_indices = (
            similarity_scores
            .argsort()[::-1][:k]
        )

        retrieved = []

        for idx in top_indices:

            retrieved.append({
                "text":
                self.texts[idx],

                "metadata":
                self.metadata[idx]
            })

        self.last_sources = retrieved

        return retrieved


    def count_tokens(self):

        history_text = ""

        for message in self.history:

            history_text += (
                f"{message['role']}: "
                f"{message['content']}\n"
            )

        tokens = self.tokenizer.encode(
            history_text
        )

        return len(tokens)


    def trim_history(self):

        total_tokens = (
            self.count_tokens()
        )

        if total_tokens > self.token_limit:

            self.history = self.history[-8:]

            print(
                "\nHistory trimmed "
                "to last 4 exchanges."
            )


    def show_sources(self):

        print("\nSOURCES")

        for i, source in enumerate(
            self.last_sources,
            start=1
        ):

            page = (
                source["metadata"]
                .get("page", "Unknown")
            )

            preview = (
                source["text"][:100]
                .replace("\n", " ")
            )

            print(f"\nSource {i}")

            print(f"Page: {page}")

            print(
                f"Preview: {preview}..."
            )


    def summary(self):

        history_text = ""

        for message in self.history:

            history_text += (
                f"{message['role']}: "
                f"{message['content']}\n"
            )

        prompt = f"""
        Summarize this conversation
        in 3 bullet points.

        Conversation:
        {history_text}
        """

        response = self.llm.invoke(
            prompt
        )

        print("\nSUMMARY")

        print(response.content)


    def ask(self, question):

        if question.lower() == "summary":

            self.summary()

            return


        if question.lower() == "sources":

            self.show_sources()

            return


        retrieved_docs = (
            self.retrieve(question)
        )

        context = "\n".join(
            [
                doc["text"]
                for doc in retrieved_docs
            ]
        )

        history_text = ""

        for message in self.history:

            history_text += (
                f"{message['role']}: "
                f"{message['content']}\n"
            )

        prompt = f"""
        You are a helpful PDF chatbot.

        Use the context and
        conversation history
        to answer the question.

        Context:
        {context}

        History:
        {history_text}

        Question:
        {question}
        """

        response = self.llm.invoke(
            prompt
        )

        answer = response.content

        self.history.append({
            "role": "user",
            "content": question
        })

        self.history.append({
            "role": "assistant",
            "content": answer
        })

        self.trim_history()

        print("\nBOT:")

        print(answer)


def main():

    if len(sys.argv) < 2:

        print(
            "Usage: python "
            "pdf_chatbot.py file.pdf"
        )

        return


    pdf_path = "/Users/Aryaanil/Documents/survey cdc.pdf"

    groq_api_key = "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"


    chatbot = PDFChatbot(
        pdf_path,
        groq_api_key
    )

    print(
        "\nPDF Chatbot Started"
    )

    print(
        "Type 'exit' to quit"
    )

    print(
        "Type 'summary' for summary"
    )

    print(
        "Type 'sources' to view sources"
    )


    while True:

        question = input(
            "\nYOU: "
        )

        if question.lower() == "exit":

            print(
                "\nChatbot closed."
            )

            break

        chatbot.ask(question)


if __name__ == "__main__":

    main()

/var/folders/2q/c34xq8nn70lgm5zqqcds5x7m0000gn/T/ipykernel_24713/3147929531.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


No saved index found.
Creating new index...
Index saved as: /Users/Aryaanil/Documents/survey cdc_index.pkl

PDF Chatbot Started
Type 'exit' to quit
Type 'summary' for summary
Type 'sources' to view sources



YOU:  summary



SUMMARY
There is no conversation to summarize. The conversation section is empty. 

If you would like to start a conversation, I can summarize it for you later in 3 bullet points.



YOU:  sources



SOURCES



YOU:  exit



Chatbot closed.
